# 06 — Per-Variant Ensemble Evaluation (multi-seed, k=3)

Evaluates each variant's ensemble independently, aggregating across **3 seeds**.

Research question: *"does per-regime LSTM training help over a base LSTM at predicting 21-day-forward volatility?"* — answered per variant via Diebold-Mariano (baseline vs ensemble) on the **mean-of-seeds predictions**, plus a variance-across-seeds report.

## Per-seed pipeline

For each ensemble variant (O, A, B):

1. Run `src/ensemble.py --variant-suffix {base}_seed{N}` three times (once per seed) → produces 3 per-seed prediction parquets under `data/processed/seeds/`.
2. Mean-aggregate across seeds → `data/processed/test_predictions{_variant}.parquet` (filename compatible with nb 07).

Variant H is a single baseline LSTM — 3 seed predictions averaged directly via `get_aligned_predictions`.

## Output artifacts

| File | Description |
|---|---|
| `data/processed/seeds/test_predictions{_variant}_seed{N}.parquet` | Per-seed ensemble predictions (9 files for O/A/B × 3 seeds) |
| `data/processed/seeds/test_predictions_H_seed{N}.parquet` | Per-seed variant-H single-LSTM predictions (3 files) |
| `data/processed/test_predictions{_variant}.parquet` | Mean-across-seeds final prediction (used by nb 07) |

## What this notebook produces

- Per-seed metric table (one row per (variant, seed)).
- Per-variant **mean ± std** across 3 seeds (the paper's honest disclosure).
- Within-variant DM tests on the mean-of-seeds predictions (baseline vs ensemble for O / A / B).
- Each-predictor-vs-naive DM tests.
- Regime-stratified breakdown per variant.


In [1]:
import sys
import subprocess
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
import torch

REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import config
from src.ensemble import get_aligned_predictions
from src.utils import regression_metrics

DATA_PROCESSED = config.DATA_PROCESSED
MODELS_DIR = config.MODELS_DIR
LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
SEEDS_DIR = DATA_PROCESSED / "seeds"
SEEDS_DIR.mkdir(exist_ok=True)

SEEDS = [42, 43, 44]
print("Repo root:", REPO_ROOT)
print("Seeds to aggregate:", SEEDS)


Repo root: /content/repo
Seeds to aggregate: [42, 43, 44]


## Prerequisite check

Verifies all required per-seed LSTM checkpoints + HMM artifacts exist (produced by nb 03, nb 04, nb 05).


In [2]:
missing = []

# Baseline LSTMs: 4 variants × 3 seeds = 12 checkpoints
baseline_patterns = [
    ("O", "lstm_baseline_O"),
    ("A", "lstm_baseline"),
    ("H", "lstm_baseline_H"),
    ("B", "lstm_baseline_B"),
]
for _, prefix in baseline_patterns:
    for s in SEEDS:
        p = MODELS_DIR / f"{prefix}_seed{s}.pt"
        if not p.exists():
            missing.append(str(p))

# Regime LSTMs: 3 variants × 2 regimes × 3 seeds = 18 checkpoints (H excluded)
regime_patterns = [("O", "_O"), ("A", ""), ("B", "_B")]
for _, base_sfx in regime_patterns:
    for s in SEEDS:
        for regime in ["calm", "volatile"]:
            p = MODELS_DIR / f"lstm_{regime}{base_sfx}_seed{s}.pt"
            if not p.exists():
                missing.append(str(p))

# HMM artifacts per variant
hmm_paths = [
    MODELS_DIR / "hmm_meta_O.joblib", MODELS_DIR / "hmm_meta.joblib", MODELS_DIR / "hmm_meta_B.joblib",
    DATA_PROCESSED / "regime_probabilities_O.parquet",
    DATA_PROCESSED / "regime_probabilities.parquet",
    DATA_PROCESSED / "regime_probabilities_B.parquet",
]
for p in hmm_paths:
    if not p.exists():
        missing.append(str(p))

if missing:
    raise RuntimeError(
        f"Missing {len(missing)} upstream artifact(s). First 10:\n  " +
        "\n  ".join(missing[:10]) +
        ("\n  …" if len(missing) > 10 else "") +
        "\n\nRun nb 03 → nb 04 → nb 05 in order."
    )
print("All LSTM + HMM artifacts present (12 baselines + 18 regime LSTMs + 3 HMM triples).")


All LSTM + HMM artifacts present (12 baselines + 18 regime LSTMs + 3 HMM triples).


## 1. Produce per-seed ensemble predictions for variants O / A / B

Three invocations per variant (one per seed), each combining `lstm_baseline_{variant}_seed{N}` + `lstm_calm_{variant}_seed{N}` + `lstm_volatile_{variant}_seed{N}` via HMM soft probabilities. Outputs land in `data/processed/seeds/`.

The `--variant-suffix` arg is extended here to include the seed (e.g., `_O_seed42`) — `src/ensemble.py`'s suffix logic already accepts arbitrary strings, so no ensemble.py code change needed.


In [3]:
ENSEMBLE_VARIANTS = [
    {"name": "O", "base_suffix": "_O",
     "regime_probs": DATA_PROCESSED / "regime_probabilities_O.parquet",
     "hmm_meta":     MODELS_DIR / "hmm_meta_O.joblib"},
    {"name": "A", "base_suffix": "",
     "regime_probs": DATA_PROCESSED / "regime_probabilities.parquet",
     "hmm_meta":     MODELS_DIR / "hmm_meta.joblib"},
    {"name": "B", "base_suffix": "_B",
     "regime_probs": DATA_PROCESSED / "regime_probabilities_B.parquet",
     "hmm_meta":     MODELS_DIR / "hmm_meta_B.joblib"},
]

# ensemble.py writes to DATA_PROCESSED/test_predictions{suffix}.parquet.
# We want per-seed files to land in the seeds/ subdir; easiest is to run ensemble
# with suffix '_{base}_seed{N}' and then MOVE the resulting parquet into seeds/.
import shutil

for v in ENSEMBLE_VARIANTS:
    name = v["name"]
    for seed in SEEDS:
        per_seed_suffix = f"{v['base_suffix']}_seed{seed}"
        cmd = [
            sys.executable, "-u", "-m", "src.ensemble",
            "--variant-suffix", per_seed_suffix,
            "--regime-probs-path", str(v["regime_probs"]),
            "--hmm-meta-path",    str(v["hmm_meta"]),
        ]
        log_path = LOG_DIR / f"ensemble_{name}_seed{seed}.log"
        print(f"[{name} seed {seed}]  running…")
        with open(log_path, "w") as f:
            r = subprocess.run(cmd, cwd=str(REPO_ROOT), stdout=f, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            print(f"  FAILED (rc={r.returncode}) — see {log_path}")
            continue
        # ensemble wrote to DATA_PROCESSED/test_predictions{suffix}.parquet;
        # move it into the per-seed subdir.
        emitted = DATA_PROCESSED / f"test_predictions{per_seed_suffix}.parquet"
        target = SEEDS_DIR / f"test_predictions{per_seed_suffix}.parquet"
        if emitted.exists():
            shutil.move(str(emitted), str(target))
            print(f"  ok → {target.relative_to(REPO_ROOT)}")
        else:
            print(f"  WARNING: expected {emitted.name} but not found")


[O seed 42]  running…


  ok → data/processed/seeds/test_predictions_O_seed42.parquet
[O seed 43]  running…


  ok → data/processed/seeds/test_predictions_O_seed43.parquet
[O seed 44]  running…


  ok → data/processed/seeds/test_predictions_O_seed44.parquet
[A seed 42]  running…


  ok → data/processed/seeds/test_predictions_seed42.parquet
[A seed 43]  running…


  ok → data/processed/seeds/test_predictions_seed43.parquet
[A seed 44]  running…


  ok → data/processed/seeds/test_predictions_seed44.parquet
[B seed 42]  running…


  ok → data/processed/seeds/test_predictions_B_seed42.parquet
[B seed 43]  running…


  ok → data/processed/seeds/test_predictions_B_seed43.parquet
[B seed 44]  running…


  ok → data/processed/seeds/test_predictions_B_seed44.parquet


## 2. Variant H — per-seed single-LSTM predictions


In [4]:
device = torch.device("cpu")
test_df = pd.read_parquet(DATA_PROCESSED / "test.parquet")

for seed in SEEDS:
    suffix = f"_H_seed{seed}"
    pred = get_aligned_predictions("baseline", test_df, device, suffix=suffix)
    pred.name = "baseline"
    df = pred.to_frame()
    df["target"] = test_df[config.LSTM_TARGET]
    df = df.dropna()
    out_path = SEEDS_DIR / f"test_predictions{suffix}.parquet"
    df.to_parquet(out_path)
    print(f"  variant H seed {seed}: {len(df)} rows → {out_path.relative_to(REPO_ROOT)}")


  variant H seed 42: 1461 rows → data/processed/seeds/test_predictions_H_seed42.parquet
  variant H seed 43: 1461 rows → data/processed/seeds/test_predictions_H_seed43.parquet
  variant H seed 44: 1461 rows → data/processed/seeds/test_predictions_H_seed44.parquet


## 3. Mean-aggregate predictions across seeds → final prediction parquets

For each variant (O, A, B, H): load the 3 per-seed parquets, intersect on common index (some seeds may have slightly different seq_len and thus different row starts), and compute column-wise mean. Save the averaged result at `data/processed/test_predictions{_variant}.parquet` — the filename nb 07 expects.


In [5]:
def _avg_across_seeds(base_suffix, seeds=SEEDS):
    """Load per-seed prediction parquets and return a DataFrame of column-wise means."""
    dfs = []
    for s in seeds:
        p = SEEDS_DIR / f"test_predictions{base_suffix}_seed{s}.parquet"
        if not p.exists():
            print(f"  missing: {p.name}")
            continue
        dfs.append(pd.read_parquet(p))
    if not dfs:
        return None
    # Intersect indices across all per-seed dfs
    common = dfs[0].index
    for d in dfs[1:]:
        common = common.intersection(d.index)
    aligned = [d.loc[common] for d in dfs]
    # All numeric columns get averaged; p_calm/p_volatile/target should be identical across seeds.
    combined = pd.concat(aligned)
    avg = combined.groupby(level=0).mean()
    # Preserve chronological order
    avg = avg.loc[common]
    return avg

final_dfs = {}
for base_suffix, name in [("_O", "O"), ("", "A"), ("_B", "B"), ("_H", "H")]:
    avg = _avg_across_seeds(base_suffix)
    if avg is None:
        print(f"[{name}] no per-seed parquets found — skipping")
        continue
    out_path = DATA_PROCESSED / f"test_predictions{base_suffix}.parquet"
    avg.to_parquet(out_path)
    final_dfs[name] = avg
    print(f"[{name}] {len(avg)} rows → {out_path.name}")


[O] 1440 rows → test_predictions_O.parquet
[A] 1440 rows → test_predictions.parquet
[B] 1440 rows → test_predictions_B.parquet
[H] 1461 rows → test_predictions_H.parquet


## 4. Per-variant test-set metrics — mean ± std across seeds

Per-seed metrics show the variance caused by Optuna TPE + PyTorch float non-determinism. Paper should quote `mean ± std` rather than single-seed point estimates.


In [6]:
# For each variant, compute metrics on every per-seed parquet.
per_seed_metrics = []
for base_suffix, name in [("_O", "O"), ("", "A"), ("_B", "B"), ("_H", "H")]:
    for seed in SEEDS:
        p = SEEDS_DIR / f"test_predictions{base_suffix}_seed{seed}.parquet"
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        y_true = df["target"].values

        # Variant H has only 'baseline'; others have 'ensemble' too
        predictors_list = [("baseline", df["baseline"].values)]
        if "ensemble" in df.columns:
            predictors_list.append(("ensemble", df["ensemble"].values))

        for pred_name, arr in predictors_list:
            m = regression_metrics(y_true, arr)
            per_seed_metrics.append({
                "variant": name, "seed": seed, "predictor": pred_name,
                **{k: float(v) for k, v in m.items()},
            })

per_seed_df = pd.DataFrame(per_seed_metrics)
print("Per-seed test metrics (every row = one seed × one predictor):")
display(per_seed_df.set_index(["variant", "seed", "predictor"]))

# Mean ± std per (variant, predictor)
agg_rows = []
for (vname, pname), g in per_seed_df.groupby(["variant", "predictor"]):
    row = {"variant": vname, "predictor": pname, "n_seeds": len(g)}
    for metric in ("MSE", "RMSE", "MAE", "MAPE"):
        if metric in g.columns:
            row[f"{metric}_mean"] = g[metric].mean()
            row[f"{metric}_std"]  = g[metric].std(ddof=1) if len(g) > 1 else 0.0
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows).set_index(["variant", "predictor"])
print("\nMean ± std across seeds:")
display(agg_df)


Per-seed test metrics (every row = one seed × one predictor):


MSE      RMSE       MAE       MAPE
variant seed predictor                                         
O       42   baseline   0.000015  0.003822  0.002427  24.297793
             ensemble   0.000015  0.003810  0.002615  29.712988
        43   baseline   0.000021  0.004534  0.003047  31.364742
             ensemble   0.000015  0.003927  0.002759  30.458968
        44   baseline   0.000017  0.004121  0.002614  24.885544
             ensemble   0.000015  0.003906  0.002755  31.996893
A       42   baseline   0.000013  0.003655  0.002471  26.407015
             ensemble   0.000014  0.003790  0.002450  24.295042
        43   baseline   0.000015  0.003867  0.002660  29.549054
             ensemble   0.000013  0.003639  0.002449  26.613073
        44   baseline   0.000019  0.004317  0.002814  27.435756
             ensemble   0.000013  0.003592  0.002363  23.975117
B       42   baseline   0.000018  0.004228  0.002762  25.156651
             ensemble   0.000016  0.003999  0.002669  28.393834
        43   baseline   0.000016  0.003972  0.002601  27.240274
             ensemble   0.000013  0.003592  0.002379  25.032230
        44   baseline   0.000019  0.004357  0.002820  27.545147
             ensemble   0.000013  0.003552  0.002419  26.174534
H       42   baseline   0.000016  0.003981  0.002575  25.414256
        43   baseline   0.000015  0.003832  0.002545  26.620570
        44   baseline   0.000014  0.003724  0.002470  24.881171


Mean ± std across seeds:


n_seeds  MSE_mean       MSE_std  RMSE_mean  RMSE_std  \
variant predictor                                                         
A       baseline         3  0.000016  2.710292e-06   0.003946  0.000338   
        ensemble         3  0.000014  7.643940e-07   0.003674  0.000103   
B       baseline         3  0.000018  1.628021e-06   0.004186  0.000196   
        ensemble         3  0.000014  1.874678e-06   0.003714  0.000248   
H       baseline         3  0.000015  9.942426e-07   0.003846  0.000129   
O       baseline         3  0.000017  2.997782e-06   0.004159  0.000358   
        ensemble         3  0.000015  4.819907e-07   0.003881  0.000062   

                   MAE_mean   MAE_std  MAPE_mean  MAPE_std  
variant predictor                                           
A       baseline   0.002648  0.000172  27.797275  1.601913  
        ensemble   0.002421  0.000050  24.961078  1.439585  
B       baseline   0.002728  0.000113  26.647357  1.299958  
        ensemble   0.002489  0.000157  26.533533  1.709314  
H       baseline   0.002530  0.000054  25.638665  0.891150  
O       baseline   0.002696  0.000318  26.849360  3.921463  
        ensemble   0.002710  0.000082  30.722950  1.164611

## 5. WITHIN-VARIANT Diebold-Mariano tests — the core research answer

Computed on the **mean-of-seed predictions** (the averaged `test_predictions{_variant}.parquet` files). For each of O / A / B: DM(baseline, ensemble) on MSE and MAE. Positive DM ⇒ baseline has higher loss ⇒ ensemble wins.

Why use averaged predictions for DM (not per-seed):
- We're testing a property of the *model class* (ensemble vs baseline), not a single trained instance.
- Averaging reduces variance, giving a cleaner signal.
- Alternative: aggregate the per-seed DM statistics (e.g., pooled or meta-analytic). Adds complexity without proportionate rigour gain for a 3-seed design.

The per-seed MSE spread (from cell 6 above) quantifies training-stochasticity uncertainty; the DM tests here quantify comparison-between-architectures uncertainty.


In [7]:
def diebold_mariano(y_true, y_a, y_b, h=21, loss="mse"):
    y_true = np.asarray(y_true, float); y_a = np.asarray(y_a, float); y_b = np.asarray(y_b, float)
    e_a = (y_a - y_true)**2 if loss == "mse" else np.abs(y_a - y_true)
    e_b = (y_b - y_true)**2 if loss == "mse" else np.abs(y_b - y_true)
    d = e_a - e_b
    n = len(d); d_bar = float(d.mean())
    max_lag = max(h - 1, 0)
    gamma0 = float(np.var(d, ddof=0)); S = gamma0
    for k in range(1, max_lag + 1):
        w = 1.0 - k / (max_lag + 1)
        gamma_k = float(np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)))
        S += 2.0 * w * gamma_k
    if S <= 0: S = gamma0
    dm_raw = d_bar / np.sqrt(S / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm = float(dm_raw * hln)
    p  = float(2.0 * (1.0 - sp_stats.t.cdf(np.abs(dm), df=n - 1)))
    return {"dm_stat": dm, "p_value": p, "n": n}


def _sig(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "n.s."


rows = []
for vname in ["O", "A", "B"]:
    if vname not in final_dfs:
        continue
    df = final_dfs[vname]
    y_true = df["target"].values
    for loss in ["mse", "mae"]:
        r = diebold_mariano(y_true, df["baseline"].values, df["ensemble"].values, loss=loss)
        verdict = ("ensemble wins" if r["dm_stat"] > 0 else "baseline wins") if r["p_value"] < 0.05 else "tie"
        rows.append({
            "variant": vname, "comparison": "baseline vs ensemble",
            "loss": loss.upper(), "DM": round(r["dm_stat"], 3),
            "p_value": round(r["p_value"], 4), "significance": _sig(r["p_value"]),
            "verdict": verdict, "n": r["n"],
        })

within_variant_dm = pd.DataFrame(rows)
print("Within-variant DM (baseline vs ensemble on mean-of-seeds predictions):")
within_variant_dm


Within-variant DM (baseline vs ensemble on mean-of-seeds predictions):


,variant,comparison,loss,DM,p_value,significance,verdict,n
0,O,baseline vs ensemble,MSE,0.138,0.8906,n.s.,tie,1440
1,O,baseline vs ensemble,MAE,-1.790,0.0737,*,tie,1440
2,A,baseline vs ensemble,MSE,1.456,0.1457,n.s.,tie,1440
3,A,baseline vs ensemble,MAE,1.916,0.0555,*,tie,1440
4,B,baseline vs ensemble,MSE,2.168,0.0303,**,ensemble wins,1440
5,B,baseline vs ensemble,MAE,2.249,0.0247,**,ensemble wins,1440


## 6. Each predictor vs naive — sanity check

Does each variant's baseline / ensemble learn anything beyond 21-day persistence? Tested on mean-of-seed predictions.


In [8]:
naive_full = test_df["rolling_std_21"].dropna()
rows = []
for vname in ["O", "A", "H", "B"]:
    if vname not in final_dfs:
        continue
    df = final_dfs[vname]
    y_true = df["target"].values
    naive = naive_full.reindex(df.index).values
    predictors_list = [("baseline", df["baseline"].values)]
    if "ensemble" in df.columns:
        predictors_list.append(("ensemble", df["ensemble"].values))
    for pname, arr in predictors_list:
        for loss in ["mse", "mae"]:
            r = diebold_mariano(y_true, naive, arr, loss=loss)
            verdict = (f"{pname} wins" if r["dm_stat"] > 0 else "naive wins") if r["p_value"] < 0.05 else "tie"
            rows.append({
                "variant": vname, "predictor": pname, "loss": loss.upper(),
                "DM": round(r["dm_stat"], 3), "p_value": round(r["p_value"], 4),
                "significance": _sig(r["p_value"]), "verdict": verdict, "n": r["n"],
            })

pd.DataFrame(rows)


,variant,predictor,loss,DM,p_value,significance,verdict,n
0,O,baseline,MSE,1.427,0.1539,n.s.,tie,1440
1,O,baseline,MAE,2.560,0.0106,**,baseline wins,1440
2,O,ensemble,MSE,1.642,0.1008,n.s.,tie,1440
3,O,ensemble,MAE,2.186,0.0290,**,ensemble wins,1440
4,A,baseline,MSE,1.856,0.0637,*,tie,1440
5,A,baseline,MAE,3.686,0.0002,***,baseline wins,1440
6,A,ensemble,MSE,1.930,0.0538,*,tie,1440
7,A,ensemble,MAE,3.851,0.0001,***,ensemble wins,1440
8,H,baseline,MSE,1.905,0.0569,*,tie,1461
9,H,baseline,MAE,3.498,0.0005,***,baseline wins,1461


## 7. Regime-stratified per-variant breakdown (on mean-of-seeds predictions)


In [9]:
rows = []
for vname in ["O", "A", "B"]:
    if vname not in final_dfs:
        continue
    df = final_dfs[vname]
    y_true = df["target"].values
    is_strict_vol = df["p_volatile"].values > 0.8
    for subset_name, mask in [("strictly calm (p_cal>0.8)", ~is_strict_vol),
                              ("strictly volatile (p_vol>0.8)", is_strict_vol)]:
        if mask.sum() == 0: continue
        for pname in ("baseline", "ensemble"):
            m = regression_metrics(y_true[mask], df[pname].values[mask])
            rows.append({
                "variant": vname, "subset": subset_name, "n": int(mask.sum()),
                "predictor": pname, "MSE": m["MSE"], "MAE": m["MAE"],
            })
pd.DataFrame(rows).set_index(["variant", "subset", "predictor"])


n       MSE       MAE
variant subset                        predictor                          
O       strictly calm (p_cal>0.8)     baseline    572  0.000013  0.002254
                                      ensemble    572  0.000015  0.002821
        strictly volatile (p_vol>0.8) baseline    868  0.000015  0.002562
                                      ensemble    868  0.000014  0.002524
A       strictly calm (p_cal>0.8)     baseline   1413  0.000013  0.002404
                                      ensemble   1413  0.000012  0.002264
        strictly volatile (p_vol>0.8) baseline     27  0.000084  0.008348
                                      ensemble     27  0.000050  0.006769
B       strictly calm (p_cal>0.8)     baseline   1408  0.000014  0.002486
                                      ensemble   1408  0.000012  0.002283
        strictly volatile (p_vol>0.8) baseline     32  0.000081  0.007331
                                      ensemble     32  0.000055  0.006866

## 8. Summary (fill in after execution)

### Core research-question answer (§5 table)

Fill in from above:

| Variant | DM MSE (p_bh for 3 pairs? apply BH across these 6 tests) | DM MAE | Verdict |
|---|---|---|---|
| O | — | — | — |
| A | — | — | — |
| B | — | — | — |

Interpretation:
- All 3 tie → paper's structural-limits headline is supported on 3 independent pipelines.
- Some reject → quantify gap, note which feature-richness pipeline benefits.
- One "baseline wins" → F2 (volatile-LSTM degeneracy) actively hurts that variant's ensemble; the constant volatile output contaminates calm-day predictions.

### Variance disclosure (§4 table)

The `MSE_std / MSE_mean` ratio per variant × predictor quantifies training-stochasticity uncertainty. If >10 %, paper should explicitly say "point metrics are reported as mean ± std across 3 seeds; single-seed values can differ by a factor of 2×."

### Hand-off to nb 07

nb 07 loads the mean-of-seeds prediction parquets we wrote here (`test_predictions{_variant}.parquet`) and does the cross-variant 6-way comparison (vs naive, vs HAR-RV, pairwise DM with Bonferroni + BH-FDR correction, F1 cross-correlation, F2 prediction-std).
